## GENERATING FAKE DATA

In [15]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from faker import Faker
import random


fake = Faker('id_ID') #clone of indonesian locale
random.seed(11)  
fake.seed_instance(11)

# target number of rows
n_records = 1234

package_type = ['solo', 'family', 'honeymoon', 'group']
months = pd.date_range(start='2023-01-01', periods=12, freq='M').strftime('%Y-%m').tolist()
gender = ['male', 'female', 'other']
customer_type = ['regular', 'new']
domains = ['gmail.com', 'yahoo.com', 'hotmail.com', 'outlook.com']

# Weighted probabilities
gender_weights = [0.48, 0.48, 0.04]
customer_type_weights = [0.7, 0.3]

# Function to generate realistic age (skewed toward 25–45)
def generate_age():
    age = int(np.random.normal(loc=35, scale=10))
    return max(18, min(age, 70))  # clamp between 18 and 70

# Function to generate random name based on gender
def generate_name(g):
    if g == 'male':
        return fake.name_male()
    elif g == 'female':
        return fake.name_female()
    else:
        return fake.name()

# Function to generate email with occasional missing or typo
def generate_email(name):
    if random.random() < 0.03:
        return None  # 3% chance of missing email
    domain = random.choice(domains)
    email_base = name.lower().replace(' ', '.')
    if random.random() < 0.05:
        email_base = email_base.replace('a', '@').replace('o', '0')  # 5% chance of typo
    return f"{email_base}@{domain}"

# Generate gender first
genders = random.choices(gender, weights=gender_weights, k=n_records)

# Generate random data - 1st table: customer data
customer_names = [generate_name(g) for g in genders]

# id, name, gender, age, nationality, email, customer_type
customer_data = {
    'customer_id': [f"MY{1000 + i}" for i in range(n_records)],
    'customer_name': customer_names,
    'gender': genders,
    'age': [generate_age() for _ in range(n_records)],
    'email': [generate_email(name) for name in customer_names],
    'customer_type': random.choices(customer_type, weights=customer_type_weights, k=n_records)    
}

customers = pd.DataFrame(customer_data)
customers.head()

C:\Users\User\AppData\Local\Temp\ipykernel_9556\3516560834.py:16: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  months = pd.date_range(start='2023-01-01', periods=12, freq='M').strftime('%Y-%m').tolist()


,customer_id,customer_name,gender,age,email,customer_type
0,MY1000,Zelaya Purwanti,male,31,zelaya.purwanti@outlook.com,new
1,MY1001,"Dr. Mila Melani, M.Farm",female,46,"dr..mila.melani,.m.farm@hotmail.com",regular
2,MY1002,Cengkal Mulyani,female,33,cengkal.mulyani@yahoo.com,regular
3,MY1003,Siti Susanti,male,27,siti.susanti@yahoo.com,new
4,MY1004,"R.M. Bakiono Hassanah, S.Pd",female,45,"r.m..bakiono.hassanah,.s.pd@yahoo.com",new


In [16]:
# Generate random data - 2nd table: travel_package
# col name: package_id, package_name, package_type, duration_days, price_per_pax, category, destination

destination_malaysia = ['Kuala Lumpur', 'Penang', 'Melaka', 'Langkawi', 'Pulau Perhentian', 'Pulau Redang', 'Pulau Tioman', 'Pulau Sipadan', 'Cameron Highlands', 'Kota Kinabalu', 'Taman Negara', 'Borneo', 'Putrajaya'] 
southeast_asia_destinations = [
    "Bali","Jakarta","Yogyakarta","Bangkok","Chiang Mai","Phuket","Krabi","Hanoi","Ho Chi Minh City","Halong Bay","Sapa","Nha Trang","Luang Prabang","Vientiane","Phnom Penh","Siem Reap","Bagan","Yangon",              
    "Singapore","Manila","Cebu","Boracay","Brunei","Dili","Lombok","Komodo Island" ]

all_destinations = destination_malaysia + southeast_asia_destinations
n_package = len(all_destinations)

# Ensure each destination only appears once in the package_name
package_name = all_destinations.copy()
random.shuffle(package_name)

# Weighted probability for package_type (e.g. family and group more common)
package_type_list = ['solo', 'family', 'honeymoon', 'group']
package_type_weights = [0.2, 0.4, 0.2, 0.2]
package_type = random.choices(package_type_list, weights=package_type_weights, k=n_package)

# Function to determine base price_per_pax for each package based on type and destination
def get_price(name, type):
    if name in destination_malaysia:
        if type == 'family':
            return fake.random_int(500, 800)
        elif type == 'group': 
            return fake.random_int(800, 1000)
        elif type == 'honeymoon':
            return fake.random_int(1000, 2000)
        elif type == 'solo':
            return fake.random_int(800, 1200)
    elif name in southeast_asia_destinations:
        if type == 'family':
            return fake.random_int(1500, 1900)
        elif type == 'group':
            return fake.random_int(1700, 2700)
        elif type == 'honeymoon':
            return fake.random_int(2500, 3500)
        elif type == 'solo':
            return fake.random_int(1700, 3000)

# Function to generate category based on type and destination
def get_category(type, name):
    beachy = ['Langkawi', 'Pulau Perhentian', 'Pulau Redang', 'Pulau Tioman', 'Pulau Sipadan', 
              'Bali', 'Phuket', 'Krabi', 'Boracay', 'Nha Trang', 'Komodo Island']
    if type == 'honeymoon':
        return 'beach'
    elif name in beachy:
        return 'eco'
    else:
        return random.choice(['adventure', 'cultural'])

# Slightly longer durations for international destinations
def get_duration(name):
    if name in destination_malaysia:
        return fake.random_int(min=2, max=5)
    else:
        return fake.random_int(min=4, max=7)

# Create values
price_per_pax = [get_price(name, type) for name, type in zip(package_name, package_type)]
duration_days = [get_duration(name) for name in package_name]
category = [get_category(type, name) for type, name in zip(package_type, package_name)]

# Final assembly
travel = {
    'package_id': [f"PC{10 + i}" for i in range(n_package)],
    'package_name': package_name,
    'package_type': package_type,
    'duration_days': duration_days,
    'price_per_pax': price_per_pax,
    'category': category,
    'destination': package_name
}

travel_package = pd.DataFrame(travel)
travel_package.head()


,package_id,package_name,package_type,duration_days,price_per_pax,category,destination
0,PC10,Manila,solo,6,2387,adventure,Manila
1,PC11,Krabi,group,4,2256,eco,Krabi
2,PC12,Taman Negara,solo,2,1018,adventure,Taman Negara
3,PC13,Putrajaya,family,3,763,adventure,Putrajaya
4,PC14,Cebu,family,5,1784,cultural,Cebu


In [17]:
# Generate data - 3rd table: Travel_guide
# staff_id, staff_name, role, working_status, commission_rate

guide_name = ['Nasrul', 'Khair', 'Intan', 'Amira', 'Nuha', 'Alissa', 'Waddah', 'Izzah', 'Nasir']
role = ['main_guide', 'assistant_guide']
working_status = ['permanent', 'contract', 'freelance']

# Use weighted selection for working_status (permanent more likely)
working_status_weights = [0.5, 0.3, 0.2]
assigned_status = random.choices(working_status, weights=working_status_weights, k=len(guide_name))

# Use weighted role selection — more assistants than mains
role_weights = [0.3, 0.7]
assigned_roles = random.choices(role, weights=role_weights, k=len(guide_name))

# Function to determine commission rate
def get_comission_rate(working_status):
    if working_status == 'freelance':
        return fake.random_int(10, 15) / 100  
    elif working_status == 'contract':
        return fake.random_int(7, 13) / 100
    else:
        return fake.random_int(5, 10) / 100

staff_data = {
    'staff_id': [str(i).zfill(4) for i in range(101, 101 + len(guide_name))],
    'staff_name': guide_name,
    'role': assigned_roles,
    'working_status': assigned_status
}

travel_guide = pd.DataFrame(staff_data)

# commission_rate column
travel_guide['comission_rate'] = travel_guide['working_status'].apply(get_comission_rate)

# Preview
travel_guide.head()


,staff_id,staff_name,role,working_status,comission_rate
0,0101,Nasrul,main_guide,contract,0.08
1,0102,Khair,assistant_guide,contract,0.11
2,0103,Intan,assistant_guide,freelance,0.12
3,0104,Amira,main_guide,freelance,0.12
4,0105,Nuha,main_guide,contract,0.08


In [18]:
# Generate random data - 4th table: sales

# listing customer_id and package_id for later use
customer_id = list(customers['customer_id'])
package_id = list(travel_package['package_id'])
guide_id = list(travel_guide['staff_id'])

# Create mapping: package_id → package_type
package_type_map = travel_package.set_index('package_id')['package_type'].to_dict()

# Payment options
payment_status = ['paid', 'pending', 'refunded']
payment_method = ['credit_card', 'online_transfer', 'cash', 'eWallet']
payment_method_weights = [0.4, 0.3, 0.2, 0.1]  # weighted for realism
channel_options = ['online', 'offline', 'agent', 'walk in']
channel_weights = [0.5, 0.2, 0.2, 0.1]  # assume most come from online

# Function to determine num_of_pax based on package_type
def get_pax(package_type):
    ptype = package_type.lower()
    if ptype == 'family':
        return fake.random_int(3, 7)
    elif ptype == 'honeymoon':
        return 2
    elif ptype == 'group':
        return fake.random_int(6, 18)
    else:
        return 1

# Number of records
total_sales_record = 1234  # same as customer records for simplicity

# Random assignments
booking_customer_ids = [random.choice(customer_id) for _ in range(total_sales_record)]
booking_package_ids = [random.choice(package_id) for _ in range(total_sales_record)]
booking_package_types = [package_type_map[pid] for pid in booking_package_ids]
num_of_pax = [get_pax(ptype) for ptype in booking_package_types]

# More realistic booking dates — more bookings during peak months
def generate_booking_date():
    month_weights = [1, 0.8, 0.9, 1.1, 1.2, 1.5, 1.7, 1.6, 1.2, 1, 0.9, 0.8]  # peak mid-year
    months = pd.date_range('2023-01-01', periods=12, freq='MS')
    chosen_month = random.choices(months, weights=month_weights, k=1)[0]
    day = fake.random_int(1, 28)
    return pd.Timestamp(chosen_month.replace(day=day))

booking_dates = [generate_booking_date() for _ in range(total_sales_record)]

# Travel date = 1 to 90 days after booking
travel_dates = [bd + pd.Timedelta(days=random.randint(1, 90)) for bd in booking_dates]

# More realistic discounts (most get < 5%)
def get_discount_per_booking(pax, price_per_pax):
    gross = pax * price_per_pax
    percent = random.choices([0, 0.05, 0.1, 0.15], weights=[0.4, 0.4, 0.15, 0.05])[0]
    return round(gross * percent)

# Payment status logic: travel already occurred → more likely to be 'paid' or 'refunded'
today = pd.Timestamp('2024-01-01')  # use fixed today for reproducibility

def get_payment_status(travel_date):
    if travel_date < today:
        return random.choices(['paid', 'refunded'], weights=[0.85, 0.15])[0]
    else:
        return random.choices(['paid', 'pending'], weights=[0.3, 0.7])[0]

# Final booking dataset
booking_data = {
    'booking_id': [f"BK{100 + i}" for i in range(total_sales_record)],
    'customer_id': booking_customer_ids,
    'package_id': booking_package_ids,
    'guide_id': [random.choice(guide_id) for _ in range(total_sales_record)],
    'booking_date': booking_dates,
    'travel_date': travel_dates,
    'num_of_pax': num_of_pax,
    'discount_promotion': [get_discount_per_booking(p, travel_package.loc[travel_package['package_id'] == pid, 'price_per_pax'].values[0])for p, pid in zip(num_of_pax, booking_package_ids)],
    'channel': random.choices(channel_options, weights=channel_weights, k=total_sales_record),
    'payment_status': [get_payment_status(td) for td in travel_dates],
    'payment_method': random.choices(payment_method, weights=payment_method_weights, k=total_sales_record)
}

# Create DataFrame
sales = pd.DataFrame(booking_data)
sales.head()


,booking_id,customer_id,package_id,guide_id,booking_date,travel_date,num_of_pax,discount_promotion,channel,payment_status,payment_method
0,BK100,MY1525,PC37,0105,2023-06-17,2023-07-10,2,645,online,paid,credit_card
1,BK101,MY1108,PC27,0109,2023-08-13,2023-09-11,4,0,offline,paid,online_transfer
2,BK102,MY2042,PC18,0106,2023-06-26,2023-08-24,17,1895,online,paid,cash
3,BK103,MY1263,PC26,0101,2023-12-27,2024-01-18,4,230,agent,pending,credit_card
4,BK104,MY1229,PC32,0106,2023-04-23,2023-06-19,12,0,agent,paid,cash


In [19]:
# Generate random data - 5th table: feedback

# Feedback comments (grouped by tone)
positive_comments = [
    "The hotel and guide exceeded our expectations!",
    "Everything was well organized and smooth.",
    "Truly a memorable adventure for our family!",
    "Beautiful destinations and a relaxing itinerary.",
    "Highly recommend this package to others."
]

negative_comments = [
    "Some meals weren’t up to standard.",
    "Overall good, but the hotel could be better.",
    "Tour guide was late and disorganized.",
    "The schedule was a bit tight but manageable.",
    "Too many hidden charges in the package."
]

# Logic for selecting appropriate comments
def select_comment(score):
    if score >= 7:
        return random.choice(positive_comments)
    elif score <= 4:
        return random.choice(negative_comments)
    else:
        return random.choice(positive_comments + negative_comments)

# Recommendation logic
def recommendation(score):
    if score >= 7:
        return 'Yes'
    elif score < 4:
        return 'No'
    else:
        return random.choice(['Yes', 'No'])

# Slight skew: more scores around 7–9, some lower
def skewed_score():
    weights = [2, 2, 3, 5, 8, 10, 12, 15, 12, 11]  # index 0 = score 1
    return random.choices(range(1, 11), weights=weights)[0]

# Generate feedback data
num_feedbacks = 320
feedback_data = []

for i in range(1, num_feedbacks + 1):
    row = sales.sample(n=1).iloc[0]
    score = skewed_score()
    travel_date = pd.to_datetime(row["travel_date"])
    feedback_date = fake.date_between(start_date=travel_date, end_date=travel_date + pd.Timedelta(days=90))

    base_rating = round(score / 2)  # Score 10 ≈ rating 5, score 6 ≈ rating 3
    feedback_data.append({
        "feedback_id": f"FB{i:03}",
        "package_id": row["package_id"],
        "booking_id": row["booking_id"],
        "satisfaction_score": score,
        "feedback_date": feedback_date,
        "comments": select_comment(score),
        "rating_hotel": min(5, max(2, base_rating + random.randint(-1, 1))),
        "rating_guide": min(5, max(2, base_rating + random.randint(-1, 1))),
        "rating_itinerary": min(5, max(2, base_rating + random.randint(-1, 1))),
        "would_recommend": recommendation(score)
    })

feedback = pd.DataFrame(feedback_data)
feedback.head()


,feedback_id,package_id,booking_id,satisfaction_score,feedback_date,comments,rating_hotel,rating_guide,rating_itinerary,would_recommend
0,FB001,PC25,BK752,8,2023-07-11,Highly recommend this package to others.,3,5,5,Yes
1,FB002,PC10,BK617,10,2023-11-27,The hotel and guide exceeded our expectations!,5,4,5,Yes
2,FB003,PC31,BK677,10,2023-06-24,Highly recommend this package to others.,5,4,5,Yes
3,FB004,PC21,BK481,10,2024-02-29,The hotel and guide exceeded our expectations!,5,4,5,Yes
4,FB005,PC42,BK1109,8,2024-01-31,Beautiful destinations and a relaxing itinerary.,5,4,3,Yes


In [20]:
# Generate data - 6th table: Refund
# refund_id, booking_id, refund_date, refund_amount, refund_percentage, reason

refund_reasons = [
    'Cancellation by customer',
    'Package change request',
    'Travel restrictions',
    'Unforeseen circumstances',
    'Service quality issues',
    'Booking errors',
    'Natural disasters',
    'Health emergencies',
    'Other'
]

refund_statuses = ['Pending', 'Approved', 'Rejected']

# Filter sales to only include bookings with payment_status == 'refunded'
eligible_refunds = sales[sales['payment_status'] == 'refunded'].reset_index(drop=True)

# Choose number of refunds, constrained to valid refunded bookings
refund_rows = min(30, len(eligible_refunds))  # Cap at 30 or less depending on available

# Sample from eligible sales
sampled_sales = eligible_refunds.sample(n=refund_rows).reset_index(drop=True)

# Create lookup: package price
package_price_map = travel_package.set_index('package_id')['price_per_pax'].to_dict()

# Function to get refund percentage based on reason
def get_refund_percentage(reason):
    if reason in ['Natural disasters', 'Health emergencies', 'Travel restrictions']:
        return round(random.uniform(20, 50), 2)
    elif reason in ['Cancellation by customer', 'Service quality issues']:
        return round(random.uniform(10, 30), 2)
    else:
        return round(random.uniform(5, 20), 2)

# Build refund data
refund_data = []

for i in range(refund_rows):
    booking_id = sampled_sales.loc[i, 'booking_id']
    customer_id = sampled_sales.loc[i, 'customer_id']
    booking_date = pd.to_datetime(sampled_sales.loc[i, 'booking_date'])
    travel_date = pd.to_datetime(sampled_sales.loc[i, 'travel_date'])
    package_id = sampled_sales.loc[i, 'package_id']
    num_pax = sampled_sales.loc[i, 'num_of_pax']

    price_per_pax = package_price_map.get(package_id, 1000)
    reason = random.choice(refund_reasons)
    refund_pct = get_refund_percentage(reason)

    base_amount = price_per_pax * num_pax
    refund_amount = round(base_amount * (refund_pct / 100))

    # Refund issued between booking and travel, or within 30 days post-travel
    refund_date = fake.date_between(start_date=booking_date, end_date=travel_date + pd.Timedelta(days=30))
    status = random.choices(['Approved', 'Pending', 'Rejected'], weights=[0.7, 0.2, 0.1])[0]

    refund_data.append({
        'refund_id': f'RF{i+1:03}',
        'booking_id': booking_id,
        'customer_id': customer_id,
        'refund_date': refund_date,
        'refund_amount': refund_amount,
        'refund_percentage': refund_pct,
        'reason': reason,
        'refund_status': status
    })

refund = pd.DataFrame(refund_data)
refund.head()


,refund_id,booking_id,customer_id,refund_date,refund_amount,refund_percentage,reason,refund_status
0,RF001,BK273,MY1686,2023-04-03,459,9.95,Unforeseen circumstances,Pending
1,RF002,BK1134,MY1546,2023-07-20,227,22.26,Health emergencies,Rejected
2,RF003,BK353,MY1561,2023-03-10,335,17.40,Other,Pending
3,RF004,BK1116,MY1537,2023-09-05,3369,49.73,Natural disasters,Pending
4,RF005,BK293,MY1729,2023-10-25,586,11.78,Unforeseen circumstances,Pending


## TRANSFORMATION

In [21]:
# 1. customers table
# clean up the email column
customers['email'] = customers['email'].str.replace(',','').str.replace('..','.')
customers.isna().sum()

# fill na email with 'not provided'
customers['email'] = customers['email'].fillna('not_provided')

In [22]:
# 2. feedback table

feedback.dtypes

feedback['feedback_date'] = pd.to_datetime(feedback['feedback_date'])
feedback.head()

,feedback_id,package_id,booking_id,satisfaction_score,feedback_date,comments,rating_hotel,rating_guide,rating_itinerary,would_recommend
0,FB001,PC25,BK752,8,2023-07-11,Highly recommend this package to others.,3,5,5,Yes
1,FB002,PC10,BK617,10,2023-11-27,The hotel and guide exceeded our expectations!,5,4,5,Yes
2,FB003,PC31,BK677,10,2023-06-24,Highly recommend this package to others.,5,4,5,Yes
3,FB004,PC21,BK481,10,2024-02-29,The hotel and guide exceeded our expectations!,5,4,5,Yes
4,FB005,PC42,BK1109,8,2024-01-31,Beautiful destinations and a relaxing itinerary.,5,4,3,Yes


In [23]:
# 3. refund table

refund['refund_date'] = pd.to_datetime(refund['refund_date'])
refund.dtypes

refund_id                    object
booking_id                   object
customer_id                  object
refund_date          datetime64[ns]
refund_amount                 int64
refund_percentage           float64
reason                       object
refund_status                object
dtype: object

In [24]:
# 4. travel_guide table

travel_guide['role'] = travel_guide['role'].str.replace('_', ' ')
travel_guide.head()



,staff_id,staff_name,role,working_status,comission_rate
0,0101,Nasrul,main guide,contract,0.08
1,0102,Khair,assistant guide,contract,0.11
2,0103,Intan,assistant guide,freelance,0.12
3,0104,Amira,main guide,freelance,0.12
4,0105,Nuha,main guide,contract,0.08


In [25]:
# 5. travel_package table
travel_package.dtypes

package_id       object
package_name     object
package_type     object
duration_days     int64
price_per_pax     int64
category         object
destination      object
dtype: object

In [26]:
# 6. sales table

sales['payment_method'] = sales['payment_method'].str.replace('_', ' ')
sales.head()

,booking_id,customer_id,package_id,guide_id,booking_date,travel_date,num_of_pax,discount_promotion,channel,payment_status,payment_method
0,BK100,MY1525,PC37,0105,2023-06-17,2023-07-10,2,645,online,paid,credit card
1,BK101,MY1108,PC27,0109,2023-08-13,2023-09-11,4,0,offline,paid,online transfer
2,BK102,MY2042,PC18,0106,2023-06-26,2023-08-24,17,1895,online,paid,cash
3,BK103,MY1263,PC26,0101,2023-12-27,2024-01-18,4,230,agent,pending,credit card
4,BK104,MY1229,PC32,0106,2023-04-23,2023-06-19,12,0,agent,paid,cash


In [ ]:
# Saving all csv files
customers.to_csv(r"C:\Users\User\Desktop\Data Analyst\End To End Project\PPNJ\data_creation\dim_customer.csv", index=False)
travel_package.to_csv(r"C:\Users\User\Desktop\Data Analyst\End To End Project\PPNJ\data_creation\dim_travel_package.csv", index=False)            
sales.to_csv(r"C:\Users\User\Desktop\Data Analyst\End To End Project\PPNJ\data_creation\fact_sales.csv", index=False)
feedback.to_csv(r"C:\Users\User\Desktop\Data Analyst\End To End Project\PPNJ\data_creation\dim_feedback.csv", index=False)
travel_guide.to_csv(r"C:\Users\User\Desktop\Data Analyst\End To End Project\PPNJ\data_creation\dim_travel_guide.csv", index=False)
refund.to_csv(r"C:\Users\User\Desktop\Data Analyst\End To End Project\PPNJ\data_creation\dim_refund.csv", index=False)